# ⌨️ Chapter 3: Working with the Command Line

📖 Book: Chapter 3, pp. 112–149 · ⏱️ 2–3 hours · 🧱 Needs: Ch 1–2 · Level: Intermediate

---

> **🎯 Key idea:** `subprocess.run()` with `shell=False` and `argparse`/`click` lets Python wrap system binaries safely without command-injection risks.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `os.popen()`, `universal_newlines=True`  
> **🆕 Now (2026):** `subprocess.run(..., text=True, check=True)`  
> **Why:** Safer, captures `stdout`/`stderr` & exit codes

> **📜 Then (book, ~2020):** `sys.argv` / `optparse`  
> **🆕 Now (2026):** `argparse` (stdlib) / `click` / `fire`  
> **Why:** Built-in `--help`, validation, subcommands

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
You need a CLI tool that checks `git` status, runs health checks, and accepts `--env prod` flags cleanly. Using `os.system()` or `shell=True` with user input opens a command-injection hole, and `sys.argv[1]` breaks as soon as flags grow. Here you learn safe `subprocess.run()` execution, `argparse` subcommands, and runtime plugins.

### Concept: Safe `subprocess.run()`
**What:** Spawns child OS processes  
**Why:** Wraps `git`/`docker`/`kubectl` safely  
**How:** Passes arg list to `execve` without `/bin/sh`  
**When:** Only when stdlib (`pathlib`/`shutil`) cannot do it

📖 *Book listing:* `subprocess.run` Maritime Wrapper (pp. 115–120)

**Plain-language analogy:** `subprocess.run()` with `shell=False` is like handing a pre-filled form directly to a clerk; `shell=True` is like reading instructions aloud where a prankster can shout extra commands.

**Walkthrough:** The book invokes external OS binaries by passing an argument list (`['ls', '-l']`) with `capture_output=True` and `check=True`. Keeping `shell=False` blocks shell-injection attacks.

### Concept: Dynamic Plugins (`pkgutil`)
**What:** Runtime module discovery  
**Why:** Extends CLIs without hardcoded imports  
**How:** Scans `sys.path` + `importlib.import_module()`  
**When:** Extensible DevOps toolkits

📖 *Book listing:* `pkgutil` & `importlib` Plugin Discovery (pp. 136–139)

**Plain-language analogy:** A plugin loader (`pkgutil` + `importlib`) is like a USB port: your main CLI doesn't need to know every device upfront—it scans what's plugged in and calls `run()`.

**Walkthrough:** The book scans `sys.path` with `pkgutil.iter_modules()`, filters modules by name prefix, loads each via `importlib.import_module()`, and calls a standardized `run()` hook.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: sys, os.environ, and subprocess.run()
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: sys, os.environ, and subprocess.run()
import os, subprocess, sys

print('Platform:', sys.platform, '| Python:', sys.version.split()[0])
cp = subprocess.run(['python3', '-c', 'print("hello child")'], capture_output=True, text=True, timeout=5, check=True)
print('rc:', cp.returncode, '| stdout:', cp.stdout.strip())

Platform: linux | Python: 3.13.14
rc: 0 | stdout: hello child


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 2: Beginner Example 2: argparse flags & defaults
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: argparse flags & defaults
import argparse

parser = argparse.ArgumentParser(description='Deploy helper')
parser.add_argument('service')
parser.add_argument('--env', choices=['dev', 'staging', 'prod'], default='dev')
args = parser.parse_args(['api', '--env', 'staging'])
print(f'Deploying {args.service} -> {args.env}')

Deploying api -> staging


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: Reading env vars with defaults
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: Reading env vars with defaults
import os
region = os.environ.get('AWS_REGION', 'us-east-1')
print('Target region:', region)

Target region: us-east-1


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Subcommands, Plugins & `click`

#### Example 4: Intermediate Example 1: argparse subcommands
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: argparse subcommands
import argparse, subprocess

p = argparse.ArgumentParser(prog='devops-ctl')
sub = p.add_subparsers(dest='cmd', required=True)
sub.add_parser('py-version')
ns = p.parse_args(['py-version'])
if ns.cmd == 'py-version':
    print(subprocess.run(['python3', '--version'], capture_output=True, text=True, check=True).stdout.strip())

Python 3.13.14


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 5: Intermediate Example 2: Dynamic plugin loader (pkgutil + importlib)
*Expected output:* Printed labelled output below.

In [5]:
# Intermediate Example 2: Dynamic plugin loader (pkgutil + importlib)
import importlib, pkgutil, sys, tempfile
from pathlib import Path

def load_plugins(plugin_dir: str, prefix: str = 'plugin_') -> list[str]:
    out: list[str] = []
    sys.path.insert(0, plugin_dir)
    try:
        for _, name, _ in pkgutil.iter_modules([plugin_dir]):
            if name.startswith(prefix):
                mod = importlib.import_module(name)
                if callable(getattr(mod, 'run', None)):
                    out.append(mod.run())
    finally:
        sys.path.pop(0)
    return out

with tempfile.TemporaryDirectory() as tmp:
    (Path(tmp) / 'plugin_disk.py').write_text("def run(): return 'disk: OK'")
    print('Plugins:', load_plugins(tmp))

Plugins: ['disk: OK']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 6: External: Click & Fire CLI frameworks (requires: pip install click fire)
*Expected output:* Simulated fallback if optional package is missing.

In [6]:
# External: Click & Fire CLI frameworks (requires: pip install click fire)
try:
    import click
    print('click installed:', click.__version__)
except ImportError:
    print('click not installed (install via: pip install click); stdlib argparse is ready!')

click not installed (install via: pip install click); stdlib argparse is ready!


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Safe Command Runner** handling `FileNotFoundError`, `TimeoutExpired`, and `CalledProcessError` cleanly.

#### Example 7: import subprocess
*Expected output:* Printed labelled output below.

In [7]:
import subprocess

def safe_exec(cmd: list[str], timeout: int = 10) -> str:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=True).stdout.strip()
    except FileNotFoundError:
        raise RuntimeError(f'Binary not found: {cmd[0]}')
    except subprocess.TimeoutExpired:
        raise RuntimeError(f'Timed out after {timeout}s: {cmd[0]}')
    except subprocess.CalledProcessError as err:
        raise RuntimeError(f'{cmd[0]} failed (exit {err.returncode}): {err.stderr.strip()}')

print('Uname:', safe_exec(['uname', '-s']))

Uname: Linux


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

#### Example 8: Real Tool Demo (guarded with shutil.which)
*Expected output:* Printed labelled output below.

In [8]:
# Real Tool Demo (guarded with shutil.which)
import shutil, subprocess
if shutil.which('git'):
    print('Real git:', subprocess.run(['git', '--version'], capture_output=True, text=True, check=True).stdout.strip())
else:
    print('git not installed on host; skipped')

Real git: git version 2.56.1


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **`shell=True` with user input** -> Command injection (`; rm -rf /`). ✅ Pass a list of strings with `shell=False`.
- ❌ **Passing `'ls -l'` as one string** -> `FileNotFoundError` when `shell=False`. ✅ Split into `['ls', '-l']`.
- ❌ **Missing `timeout=`** -> Hung child process freezes CI. ✅ Always set `timeout=10` and `check=True`.

#### Example 9: import subprocess
*Expected output:* Printed labelled output below.

In [9]:
import subprocess
try:
    subprocess.run(['nonexistent_bin_xyz'], capture_output=True, text=True, check=True)
except FileNotFoundError as err:
    print('Handled FileNotFoundError:', err)

Handled FileNotFoundError: [Errno 2] No such file or directory: 'nonexistent_bin_xyz'


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

## 10. 🛠️ Hands-On Exercises
1. **(Beginner)** `get_env_required(key: str) -> str`: Raise `KeyError` if unset.
2. **(Beginner)** `run_with_status(cmd: list[str]) -> bool`: Return `True` iff exit code == 0.
3. **(Beginner)** Parse `--host` (required) and `--port` (int, default `8080`) with `argparse`.
4. **(Intermediate)** Build an `argparse` parser with `check` and `version` subcommands.
5. **(Intermediate)** Extend `load_plugins` to skip modules missing `run()`.

## 11. 🏆 Practical DevOps Challenge
`run_diagnostics(argv: list[str]) -> dict`: Support `--json` flag, collect Python version, CWD, and `git --version` via `safe_exec`.

> **🧪 Try it:** The starter cell below reports `TODO` until you implement the function body.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement run_with_status and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [10]:
# Starter Exercise Cell (implement run_with_status and run to verify!)
def _run_starter_checks():
    assert run_with_status(['python3', '--version']) is True
def run_with_status(cmd: list[str]) -> bool:
    raise NotImplementedError('Implement run_with_status')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement run_with_status


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why is `shell=False` the safe default in `subprocess.run()`?

<details>
<summary>✅ Answer 1</summary>

Passes args directly to the OS without `/bin/sh`, preventing command injection.

</details>

**Q2:** What does `check=True` do on non-zero exit?

<details>
<summary>✅ Answer 2</summary>

Raises `subprocess.CalledProcessError`.

</details>

**Q3:** When prefer `pathlib`/`shutil` over `subprocess.run(['cp', ...])`?

<details>
<summary>✅ Answer 3</summary>

Always for file ops—native Python is faster, safer, and cross-platform.

</details>

**Q4:** Why use `if __name__ == '__main__':`?

<details>
<summary>✅ Answer 4</summary>

Prevents CLI execution when the file is imported by tests or other modules.

</details>

**Q5:** How do `pkgutil` and `importlib` work together?

<details>
<summary>✅ Answer 5</summary>

`pkgutil` discovers module names; `importlib` imports them dynamically by string name.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: `val = os.environ.get(key)`; raise `KeyError` if `None`.
- **Challenge**: Catch `RuntimeError` if `git` is missing.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [11]:
# Complete Solutions + Assert Verification
import argparse, json, os, subprocess, sys

def get_env_required(key: str) -> str:
    val = os.environ.get(key)
    if val is None:
        raise KeyError(f'Missing env var: {key}')
    return val

def run_with_status(cmd: list[str]) -> bool:
    return subprocess.run(cmd, capture_output=True, timeout=5).returncode == 0

def run_diagnostics(argv: list[str]) -> dict:
    p = argparse.ArgumentParser()
    p.add_argument('--json', action='store_true')
    opts = p.parse_args(argv)
    try:
        git_ver = safe_exec(['git', '--version'])
    except RuntimeError as e:
        git_ver = f'unavailable ({e})'
    rep = {'python': sys.version.split()[0], 'cwd': os.getcwd(), 'git': git_ver}
    if opts.json:
        print(json.dumps(rep))
    return rep

assert run_with_status(['python3', '--version']) is True
assert 'python' in run_diagnostics(['--json'])
print('All Chapter 3 assertions PASSED!')

{"python": "3.13.14", "cwd": "/home/vannaboth/Documents/python-devops/chapters/03_command_line", "git": "git version 2.56.1"}
All Chapter 3 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `subprocess.run(['cmd'], capture_output=True, text=True, check=True)` — Safe binary execution (e.g. `cp.stdout.strip()`)
- `argparse.ArgumentParser()` — Zero-dep CLI parser (e.g. `parser.add_subparsers()`)
- `pkgutil.iter_modules()` — Plugin discovery (e.g. `importlib.import_module(name)`)

### ✅ What You Can Now Do
- Execute external OS binaries safely with `subprocess.run(..., shell=False, timeout=..., check=True)`.
- Build zero-dependency subcommand CLIs with `argparse`.
- Load extensible plugins dynamically via `pkgutil` and `importlib`.

### ➡️ Next Chapter
Chapter 4 builds on this by pairing Python helpers with Linux disk, network, CPU, and syscall utilities.